In [ ]:
import pandas as pd
import json
import ast
import os  
from ast import literal_eval

INSTRUCTION = (
    "Here is a record of a user's POI accesses, your task is based on the history "
    "to predict the POI that the user is likely to access at the specified time."
)

def encode_item(item):
    if not item:
        return ""
    labels = ['a', 'b', 'c', 'd']
    if len(item) > 4:
        raise ValueError(f"Item 维度超过4: {item}")
    return "".join(f"<{labels[i]}_{val}>" for i, val in enumerate(item))

def load_pid_to_code(codebook_path):
    df_code = pd.read_csv(codebook_path)
    pid2code = {}

    for _, row in df_code.iterrows():
        pid = row["pid"]
        sid_str = row["sid"]
        try:
            item_list = literal_eval(sid_str) 
            code_str = encode_item(item_list)   # -> "<a_59><b_18><c_63>"
            pid2code[pid] = code_str
        except Exception as e:
            print(f"跳过 pid={pid}, sid={sid_str}: {e}")

    return pid2code


def convert_csv_to_llm_json_sid(
    input_file,
    output_file,
    pid2code,
    instruction=INSTRUCTION,
    keep_last_k_per_user=None,
    skip_unknown=True
):
    df = pd.read_csv(input_file)

    if keep_last_k_per_user is not None:
        df = df.groupby("UserId", group_keys=False).tail(keep_last_k_per_user).reset_index(drop=True)

    samples = []
    skipped_rows = 0

    for row in df.itertuples(index=False):
        uid = row.UserId
        try:
            poi_seq = ast.literal_eval(row.sequence_PoiId)
            time_seq = ast.literal_eval(row.sequence_UTCTimeOffset)
        except Exception as e:
            print(f"UserId={uid}: {e}")
            skipped_rows += 1
            continue

        if len(poi_seq) < 2 or len(poi_seq) != len(time_seq):
            skipped_rows += 1
            continue

        history_pids = poi_seq[:-1]
        history_times = time_seq[:-1]
        target_time = time_seq[-1]
        target_pid = poi_seq[-1]

        history_codes = []
        bad_flag = False

        for pid in history_pids:
            if pid in pid2code:
                history_codes.append(pid2code[pid])
            else:
                if skip_unknown:
                    bad_flag = True
                    break
                else:
                    history_codes.append(f"<unk_{pid}>")

        if bad_flag:
            skipped_rows += 1
            continue

        if target_pid in pid2code:
            target_code = pid2code[target_pid]
        else:
            if skip_unknown:
                skipped_rows += 1
                continue
            else:
                target_code = f"<unk_{target_pid}>"

        history_parts = [
            f"{t} visited {p}" for t, p in zip(history_times, history_codes)
        ]
        history_text = ", ".join(history_parts)

        input_text = (
            f"User_{uid} checkin history: {history_text}.\n"
            f"When {target_time} user_{uid} is likely to visit:"
        )

        samples.append({
            "instruction": instruction,
            "input": input_text,
            "output": target_code
        })

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(samples)} samples to {output_file}")
    print(f"Skipped {skipped_rows} rows")


dataset = "ca"
mode = "cvq"
codebook_path = f"data/{dataset}/{mode}_semitic_codes.csv"

pid2code = load_pid_to_code(codebook_path)

convert_csv_to_llm_json_sid(
    f"{dataset}/train_poi_sequence.csv",
    f"{dataset}/{mode}/llm_train.json",
    pid2code=pid2code,
    keep_last_k_per_user=5
)

convert_csv_to_llm_json_sid(
    f"{dataset}/validation_poi_sequence.csv",
    f"{dataset}/{mode}/llm_val.json",
    pid2code=pid2code
)

convert_csv_to_llm_json_sid(
    f"{dataset}/test_poi_sequence.csv",
    f"{dataset}/{mode}/llm_test.json",
    pid2code=pid2code
)